# 🎬 AutoDub Studio — One-Click Colab Deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedj7895-cell/ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-ElevenLabs-Quality-Open-Source-/blob/main/Colab_Runner.ipynb)

ElevenLabs-quality **open-source automatic dubbing** — Demucs v4 · Pyannote 3.1 ·
SenseVoice-Small · CosyVoice 3.0 zero-shot cloning — wrapped in an iPhone-15
frosted-glass UI.

**How to use**
1. ▸ *Runtime* ▸ *Change runtime type* ▸ **T4 GPU** ▸ Save
2. ▸ *Runtime* ▸ **Run all**
3. When the last cell finishes, click the printed **`https://….gradio.live`** link
4. In Tab 1: upload your **audio/video master + Original SRT + Translated SRT**, hit 🔍
5. Tab 2: 🧬 matches speakers & emotions (paste a HuggingFace token in Tab 1 ▸ Advanced first — see [token guide](https://github.com/syedj7895-cell/ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-ElevenLabs-Quality-Open-Source-#-one-time-hugging-face-setup-required-for-step-3))
6. Tab 3: 🚀 renders the dubbed master mix / video

Ready to use — no build step required.

In [ ]:
# ── 1 ▸ Runtime sanity check ──────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f"✅ GPU ready: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB)")
else:
    print("⚠ No GPU detected — the app still runs, but SLOWLY.")
    print("  Fix: Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save, then re-run.")

In [ ]:
# ── 2 ▸ Clone the project ─────────────────────────────────────────────
REPO_URL = ("https://github.com/syedj7895-cell/"
            "ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-"
            "ElevenLabs-Quality-Open-Source-.git")
!git clone {REPO_URL} ai-video-dubber
%cd ai-video-dubber
!ls

In [ ]:
# ── 3 ▸ Install the open-source AI frameworks ─────────────────────────
# Colab ships gradio-client 1.3.0 which crashes Gradio's /api/info endpoint
# (TypeError: argument of type 'bool' is not iterable) — force-upgrade it.
!pip install -q -U gradio-client
!pip install -q -r requirements.txt
print("✅ dependencies installed")

In [ ]:
# ── 4 ▸ Bootstrap the CosyVoice TTS engine (once · ~3 min) ────────────
# Needed for Step 7 voice cloning. Fault-tolerant: if it hiccups you can
# still run Tabs 1–2 now and re-run this cell before rendering in Tab 3.
import os, re, subprocess
if not os.path.isdir("/content/CosyVoice"):
    subprocess.run(["git", "clone", "--recursive",
                    "https://github.com/FunAudioLLM/CosyVoice.git",
                    "/content/CosyVoice"], check=False)
# Python 3.13 (Colab) fix: CosyVoice pins onnxruntime-gpu==1.18.0 which
# has no Python-3.13 wheels — relax the hard pins before installing.
_req = "/content/CosyVoice/requirements.txt"
if os.path.isfile(_req):
    _txt = open(_req, encoding="utf-8").read()
    _txt = re.sub(r"onnxruntime-gpu==[0-9.]+", "onnxruntime-gpu", _txt)
    _txt = re.sub(r"^torchaudio==.*$", "", _txt, flags=re.M)   # keep Colab's
    _txt = re.sub(r"^torch==.*$", "", _txt, flags=re.M)        # keep Colab's
    open(_req, "w", encoding="utf-8").write(_txt)
# Verbose install — capture output so failures are VISIBLE, not silent.
_cv = subprocess.run("pip install -r requirements.txt && pip install .",
                     shell=True, cwd="/content/CosyVoice",
                     capture_output=True, text=True)
if _cv.returncode != 0:
    print("⚠ CosyVoice pip install FAILED — last 2000 chars of stderr:")
    print(_cv.stderr[-2000:])
    print("→ Fix the conflict above, then re-run this cell.")
%cd /content/ai-video-dubber
try:
    from cosyvoice.cli.cosyvoice import CosyVoice2  # noqa: F401
    print("✅ CosyVoice engine ready")
except Exception as _e:
    print(f"⚠ CosyVoice not importable yet ({_e}) — re-run this cell "
          "later if Tab 3 asks for it. Tabs 1–2 work fine without it.")

In [ ]:
# ── 5 ▸ 🚀 LAUNCH — click the https://….gradio.live link printed below ─
!python app.py